## TinyShakespeare

É o dataset clássico para treinar GPTs pequenos. São peças completas de Shakespeare em inglês ~1MB de texto puro. Pequeno o suficiente para rodar local, grande o suficiente para modelo aprender algo real.

```
wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

```

ou no próprio Python antes do treino

```
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "shakespeare.txt"
)
```

## O que muda aqui em relação ao `.txt` anterior

Três coisas novas aparecem que você ainda não enfretou

**Volume real -** o arquivo tem ~40.000 linhas. Você não pode mais interar frase pr frase como antes. Precisa de `batches` - processar várias sequências ao mesmo tempo. Isso muda o treino e é o primeiro passo para escalar.


**Sequências de tamanho fixo -** em vez de tokenizar linha por linha, você vai concatenar tudo num stram contínuo de tokens e recortar em janelas de tamanho fixo. É assim que GPTs reais são treinados.


**DataLoader -** você vai usar `torch.util.data.DataLoader`, que é a forma padrão do PyTorch de iterar sobre batches, embaralhar os dados e gerenciar memória


A estrutura fica assim:
```
texto completo (40k linhas)
        ↓
stream de tokens [t1, t2, t3, t4, ... t_n]
        ↓
janelas de tamanho fixo (ex: 32 tokens cada)
[t1..t32], [t33..t64], [t65..t96], ...
        ↓
batch de 16 janelas processadas ao mesmo tempo
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import urllib.request
import re

In [ ]:
# --- 1. Dowload -----------------------------
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "shakespeare.txt"
)


# ── 2. Leitura e limpeza ──────────────────────────────────────────────────────
with open("shakespeare.txt", encoding="utf-8") as f:
    text = f.read()

# Limpeza mínima — mantém a estrutura do texto
text = text.lower()
text = re.sub(r"[^a-z\s]", "", text)   # só letras e espaços
text = re.sub(r"\s+", " ", text).strip()

words = text.split()
print(f"Total de palavras: {len(words):,}")

# ── 3. Vocab ──────────────────────────────────────────────────────────────────
vocab      = ["[PAD]", "[UNK]"] + sorted(set(words))
word2idx   = {w: i for i, w in enumerate(vocab)}
idx2word   = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Vocab size: {vocab_size:,}")

# ── 4. Stream de tokens ───────────────────────────────────────────────────────
all_tokens = [word2idx.get(w, word2idx["[UNK]"]) for w in words]
all_tokens = torch.tensor(all_tokens, dtype=torch.long)

print(f"Total de tokens: {len(all_tokens):,}")

# ── 5. Dataset de janelas deslizantes ─────────────────────────────────────────
#
#  Cada amostra é uma janela de SEQ_LEN tokens
#  O target é a mesma janela deslocada 1 posição para a direita
#
#  Ex com SEQ_LEN=4:
#  tokens:  [o, rei, e, nobre, e, justo]
#  input:   [o, rei, e, nobre]
#  target:  [rei, e, nobre, e]

SEQ_LEN = 32


Total de palavras: 202,619
Vocab size: 12,849
Total de tokens: 202,619


In [ ]:
class TextDataset(Dataset):
  def __init__(self, tokens, seq_len):
    self.tokens = tokens
    self.seq_len = seq_len

  def __len__(self):
    return len(self.tokens) - self.seq_len

  def __getitem__(self, idx):
    x = self.tokens[idx         : idx + self.seq_len]
    y = self.tokens[idx + 1     : idx + self.seq_len + 1]
    return x, y

dataset    = TextDataset(all_tokens, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Total de amostras: {len(dataset):,}")
print(f"Batches por epoch: {len(dataloader):,}")


Total de amostras: 202,587
Batches por epoch: 6,331


In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads,
                 num_layers, max_seq_len, ffn_dim):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embedding_dim)
        self.pos_emb   = nn.Embedding(max_seq_len, embedding_dim)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=0.1,  # dropout ativo agora
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.linear_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, tokens):
        seq_len = tokens.shape[1]
        x = self.token_emb(tokens) + self.pos_emb(
            torch.arange(seq_len, device=tokens.device)
        ).unsqueeze(0)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(tokens.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.linear_head(x)

In [ ]:
model = MiniGPT(
    vocab_size    = vocab_size,
    embedding_dim = 64,
    num_heads     = 4,
    num_layers    = 3,
    max_seq_len   = SEQ_LEN,
    ffn_dim       = 128,
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
print(f"\nParametros: {sum(p.numel() for p in model.parameters()):,}")



Parametros: 1,759,985


In [ ]:
# ── Treino com batches ─────────────────────────────────────────────────────────
#
#  Diferença principal: o loop interno agora é sobre o DataLoader
#  Cada iteração processa BATCH_SIZE sequências ao mesmo tempo

best_loss      = float("inf")
patience       = 3              # em epochs completas agora, não steps
patience_count = 0
best_weights   = None

for epoch in range(50):
    model.train()
    total_loss = 0

    for batch_idx, (x, y) in enumerate(dataloader):
        logits = model(x)
        loss   = F.cross_entropy(
            logits.view(-1, vocab_size),
            y.view(-1),
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # estabiliza treino
        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 500 == 0:
            print(f"  Epoch {epoch} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch {epoch} completa | Loss média: {avg_loss:.4f}\n")

    if avg_loss < best_loss - 1e-4:
        best_loss      = avg_loss
        patience_count = 0
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping na epoch {epoch}")
            break

model.load_state_dict(best_weights)

  Epoch 0 | Batch 0/6331 | Loss: 9.6042
  Epoch 0 | Batch 500/6331 | Loss: 6.8865
  Epoch 0 | Batch 1000/6331 | Loss: 6.5599
  Epoch 0 | Batch 1500/6331 | Loss: 6.3602
  Epoch 0 | Batch 2000/6331 | Loss: 6.2478
  Epoch 0 | Batch 2500/6331 | Loss: 6.1498
  Epoch 0 | Batch 3000/6331 | Loss: 5.9357
  Epoch 0 | Batch 3500/6331 | Loss: 5.9257
  Epoch 0 | Batch 4000/6331 | Loss: 5.8910
  Epoch 0 | Batch 4500/6331 | Loss: 5.9500
  Epoch 0 | Batch 5000/6331 | Loss: 5.7891
  Epoch 0 | Batch 5500/6331 | Loss: 5.7041
  Epoch 0 | Batch 6000/6331 | Loss: 5.6873

Epoch 0 completa | Loss média: 6.1156

  Epoch 1 | Batch 0/6331 | Loss: 5.5833
  Epoch 1 | Batch 500/6331 | Loss: 5.4664
  Epoch 1 | Batch 1000/6331 | Loss: 5.3648
  Epoch 1 | Batch 1500/6331 | Loss: 5.3952
  Epoch 1 | Batch 2000/6331 | Loss: 5.3689
  Epoch 1 | Batch 2500/6331 | Loss: 5.2859
  Epoch 1 | Batch 3000/6331 | Loss: 5.2805
  Epoch 1 | Batch 3500/6331 | Loss: 5.1225
  Epoch 1 | Batch 4000/6331 | Loss: 5.2568
  Epoch 1 | Batch 4500

KeyboardInterrupt: 

In [ ]:
def generate(model, prompt, max_new_tokens=5):
    model.eval()
    tokens = [word2idx.get(w, word2idx["[UNK]"]) for w in prompt.lower().split()]
    tokens = torch.tensor([tokens])

    result = prompt.split()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            inp        = tokens[:, -SEQ_LEN:]
            logits     = model(inp)
            probs      = F.softmax(logits[:, -1, :], dim=-1)
            next_tok   = torch.multinomial(probs, num_samples=1)
            result.append(idx2word[next_tok.item()])
            tokens     = torch.cat([tokens, next_tok], dim=1)

    return " ".join(result)

print("\n── Geração ──")
prompts = ["We are accounted poor citizens", "machine learning is"]
for p in prompts:
    print(f"\n  '{p}'")
    print(f"  → {generate(model, p)}")


── Geração ──

  'We are accounted poor citizens'
  → We are accounted poor citizens by this hand is it

  'machine learning is'
  → machine learning is and you know not thy


Três coisas para obsercar quando rodar:

**O tempo por epoch  -** vai ser muito maior que antes. Você tem milhares de batches por epoch agora. Isso é ML de verdade - você vai querer GPU eventualmente.

`clip_grad_norm_` - linha nova no treino. Com dataset grandes os gradientes podem explodir e desestabilizar tudo. Essa linha limita o tamanho máximo do gradiente. É  prática padrão em todo treino de transformer.

**A qualidade da geração ** - com muito mais dados o modelo vai produzir inglês reconhecível, mesmo sendo pequeno. Você vai ser a diferença de ter o dados suficientes.